# utils.ipynb - 强化学习通用工具库

提供经验回放、神经网络基类、训练曲线绘制等通用组件，被其他 Notebook 复用。

> 本 Notebook 配套《强化学习全面教程》PDF 使用。运行前请先执行 `pip install torch gymnasium numpy matplotlib tqdm`。

In [ ]:
# 本单元导入本教程通用工具与第三方库
import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 中文字体配置（Linux 路径，macOS 用户请改用 Heiti SC 或 PingFang SC）
import os, platform
if platform.system() == 'Linux' and os.path.exists('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf'):
    fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# PyTorch 设备选择
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch 版本: {torch.__version__}')
print(f'使用设备: {device}')
print(f'Gymnasium 版本: {gym.__version__}')


## 1. 经验回放（Experience Replay）

经验回放是 DQN 系列算法的核心组件，用于打破数据时间相关性、提高样本利用率。

- **标准版**：均匀采样
- **优先版（PER）**：按 |TD 误差| 比例采样

In [ ]:
import random
from collections import deque, namedtuple
from typing import Tuple

Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    """标准经验回放：均匀采样。"""
    def __init__(self, capacity: int = 10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args) -> None:
        self.buffer.append(Transition(*args))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self) -> int:
        return len(self.buffer)


class PrioritizedReplayBuffer:
    """优先经验回放（PER）。"""
    def __init__(self, capacity: int = 10_000, alpha: float = 0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = [None] * capacity
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.size = 0

    def push(self, *args, prio: float = 1.0) -> None:
        self.buffer[self.pos] = Transition(*args)
        self.priorities[self.pos] = prio
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int, beta: float = 0.4):
        probs = self.priorities[:self.size] ** self.alpha
        probs = probs / probs.sum()
        idx = np.random.choice(self.size, batch_size, p=probs)
        samples = [self.buffer[i] for i in idx]
        weights = (self.size * probs[idx]) ** (-beta)
        weights = weights / weights.max()
        return idx, Transition(*zip(*samples)), torch.tensor(weights, dtype=torch.float32)

    def update_priorities(self, idx, priorities) -> None:
        for i, p in zip(idx, priorities):
            self.priorities[i] = abs(p) + 1e-5

# 简单自检
buf = ReplayBuffer(100)
buf.push([0, 0], 0, 1.0, [1, 0], False)
print(f'ReplayBuffer 自检: len={len(buf)}, 样本={buf.sample(1)}')

## 2. 神经网络基类

包含 Q 网络、Dueling Q 网络、策略网络、价值网络。

In [ ]:
import torch.nn as nn

class QNetwork(nn.Module):
    """标准 Q 网络：MLP。"""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    def forward(self, x):
        return self.net(x)


class DuelingQNetwork(nn.Module):
    """Dueling Q 网络：V(s) 与 A(s,a) 分支。"""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.feature = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU())
        self.value = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.advantage = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, action_dim))
    def forward(self, x):
        h = self.feature(x)
        v = self.value(h)
        a = self.advantage(h)
        return v + (a - a.mean(dim=1, keepdim=True))


class PolicyNetwork(nn.Module):
    """策略网络：离散输出 logits，连续输出 (mean, std)。"""
    def __init__(self, state_dim, action_dim, hidden=128, continuous=False):
        super().__init__()
        self.continuous = continuous
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        if continuous:
            self.mean = nn.Linear(hidden, action_dim)
            self.log_std = nn.Parameter(torch.zeros(action_dim))
        else:
            self.head = nn.Linear(hidden, action_dim)
    def forward(self, x):
        h = self.net(x)
        if self.continuous:
            mean = self.mean(h)
            std = self.log_std.exp().expand_as(mean)
            return mean, std
        return self.head(h)


class ValueNetwork(nn.Module):
    """状态价值网络 V(s)。"""
    def __init__(self, state_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

print('神经网络基类定义完成 ✓')

## 3. 训练曲线绘制与通用工具

封装 matplotlib 绘图、软更新、随机种子等常用工具。

In [ ]:
def plot_rewards(rewards, title, window=20, save_path=None):
    """绘制训练奖励曲线 + 滑动平均。"""
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(rewards, alpha=0.3, color='#887129', label='每回合奖励')
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
        ax.plot(range(window - 1, len(rewards)), ma,
                color='#a2524b', linewidth=2, label=f'滑动平均 (w={window})')
    ax.set_xlabel('回合'); ax.set_ylabel('累计奖励')
    ax.set_title(title); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    if save_path:
        fig.savefig(save_path, dpi=120)
    plt.show()


def soft_update(target, source, tau=0.005):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tau * sp.data + (1 - tau) * tp.data)


def hard_update(target, source):
    target.load_state_dict(source.state_dict())


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensor(x, dtype=torch.float32, device='cpu'):
    return torch.as_tensor(np.asarray(x), dtype=dtype, device=device)

print('通用工具定义完成 ✓')